# 01 — Index and Search

The first half of the funnel: define a schema, index real products, and watch
**lexical (BM25)** and **neural (k-NN)** retrieval disagree.

**Prerequisites** — from the repo root:

```bash
docker compose -f docker/docker-compose.yml up -d
.venv/bin/python -m opensearch_demo.amazon          # downloads the Appliances corpus (~272 MB)
.venv/bin/python -m opensearch_demo.demo --dataset amazon --rebuild "warm up"   # embeds + indexes (~20 min, once)
```

Corpus: the **Appliances** category of
[McAuley-Lab/Amazon-Reviews-2023](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023)
— 94k products with title, features, description, price, rating, store, and a category tree.

In [ ]:
import json, pandas as pd
from opensearch_demo import get_client, wait_for_cluster
from opensearch_demo.search import lexical_search, neural_search

INDEX = "products-appliances"
client = get_client()
wait_for_cluster(client)
n = client.count(index=INDEX)["count"]
print(f"cluster ok — {n:,} products in '{INDEX}'")
assert n > 0, 'Index is empty — run the --rebuild command from the cell above first.'

## The schema is a set of decisions, not a formality

Every field earns its place: analysed `text` fields feed BM25, `keyword` fields
feed exact filters, numerics feed ranges, and the `knn_vector` carries HNSW
parameters that are **fixed at build time** (`m`, `ef_construction`) — only
`ef_search` can be changed later without reindexing.

In [ ]:
from opensearch_demo.schema import build_product_mapping
print(json.dumps(build_product_mapping(), indent=2))

## Lexical vs neural — the same query, two worldviews

BM25 matches **tokens**. The embedding matches **meaning**. On a real corpus
they disagree constantly, and each is right in different places.

In [ ]:
def compare(query, k=5, **filters):
    lex = lexical_search(client, query, k=k, index=INDEX, **filters)
    vec = neural_search(client, query, k=k, index=INDEX, **filters)
    rows = []
    for i in range(max(len(lex), len(vec))):
        rows.append({
            "rank": i + 1,
            "lexical (BM25)": lex[i]["title"][:70] if i < len(lex) else "",
            "neural (k-NN)":  vec[i]["title"][:70] if i < len(vec) else "",
        })
    return pd.DataFrame(rows).set_index("rank")

# Exact vocabulary: both should cope, BM25 for the obvious reason
compare("countertop nugget ice maker")

In [ ]:
# Paraphrase: describe the *problem*, not the product.
# BM25 has almost no tokens to grab; the embedding still lands in the right aisle.
compare("keep drinks cold in a dorm room without a big fridge")

In [ ]:
# Identifier: a model number appears in exactly the right documents' text.
# Embeddings blur identifiers into their neighbourhood — BM25 nails them.
compare("BPACT08WT")

## Filters: `must` narrows the world, `should` nudges the ranking

A `must` filter is membership — fail it and the document is gone. A `should`
is preference — fail it and you only lose score. For k-NN, our filters are
applied **during graph traversal** (inside the `knn` clause), so a selective
filter still returns k results instead of silently fewer.

In [ ]:
# Hard constraints: an ice maker under $150, from a store we trust the ratings of
r = neural_search(client, "portable ice maker", k=5, index=INDEX,
                  must={"price": {"gte": 1, "lte": 150},
                        "rating_number": {"gte": 100}})
pd.DataFrame([{ "title": d["title"][:60], "price": d.get("price"),
                "rating": d.get("average_rating"), "n_ratings": d.get("rating_number"),
                "category": (d.get("categories") or ["?"])[-1]} for d in r])

In [ ]:
# The category *tree* is a keyword array — a filter matches any level.
r = lexical_search(client, "compact refrigerator", k=5, index=INDEX,
                   must={"categories": "Ice Makers"})   # steer a fridge query into one aisle
pd.DataFrame([{ "title": d["title"][:70], "categories": " > ".join(d.get("categories") or [])} for d in r])

## Truncation is silent — count it or it didn't happen

Our embedding model reads **256 tokens** and discards the rest without a word.
38% of this corpus runs long. That is why `build_embedding_text` puts
title → features → description in that order: the tail gets eaten first.

In [ ]:
from opensearch_demo.amazon import load
from opensearch_demo.embed import build_embedding_text, get_encoder

docs = load(limit=2000)
tok = get_encoder().tokenizer
lens = pd.Series([len(tok.encode(build_embedding_text(d))) for d in docs])
print(f"median {lens.median():.0f} tokens | over the 256 window: {(lens > 256).mean():.0%}")
lens.hist(bins=60); None